# FPN: Feature Pyramid Network

**Paper**: Lin et al., CVPR 2017 — *Feature Pyramid Networks for Object Detection*

## The Multi-Scale Detection Problem

Objects appear at vastly different scales in images — a person far away vs close up. Traditional approaches:
- **Image pyramid**: run detector on resized copies → slow, memory heavy
- **Single feature map** (YOLO v1, Faster R-CNN): use one scale → misses small objects
- **SSD-style**: independent predictions from multiple CNN stages → weak semantics at low levels

## FPN: Top-Down Path + Lateral Connections

<img src='./figures/FPN.png' width='700'/>

FPN creates a **feature pyramid** by:
1. **Bottom-up pathway**: standard CNN forward pass (e.g., ResNet) — builds semantically rich but spatially coarse features
2. **Top-down pathway**: upsample from the deepest layer (richest semantics) back to higher resolutions
3. **Lateral connections**: 1×1 conv on each bottom-up stage + element-wise addition with the upsampled top-down features

Result: every pyramid level has **strong semantics + full resolution** — the best of both worlds.

```
ResNet Stage     FPN Level    Scale (for 800px input)
C2 (1/4)    →   P2           200x200  ← small objects
C3 (1/8)    →   P3           100x100
C4 (1/16)   →   P4            50x50
C5 (1/32)   →   P5            25x25
               P6 (pool P5)   13x13  ← large objects
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.ops import FeaturePyramidNetwork
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import requests
from io import BytesIO

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## Bottom-Up and Top-Down Pathways

<img src='./figures/bottomup.jpeg' width='350' style='display:inline'/>
<img src='./figures/topdown.png' width='350' style='display:inline'/>

The **lateral connection** at each level:
1. Apply 1×1 conv to reduce bottom-up channels to d=256
2. Upsample top-down feature by 2× (nearest-neighbor)
3. Add them element-wise
4. Apply 3×3 conv to smooth aliasing artifacts

<img src='./figures/lateralconnection.png' width='450'/>

## FPN Implementation from Scratch

In [ ]:
class LateralBlock(nn.Module):
    """1x1 lateral + 3x3 smooth for one FPN level."""
    def __init__(self, in_ch, out_ch=256):
        super().__init__()
        self.lateral = nn.Conv2d(in_ch, out_ch, 1)
        self.smooth  = nn.Conv2d(out_ch, out_ch, 3, padding=1)

    def forward(self, bottom_up, top_down=None):
        lat = self.lateral(bottom_up)
        if top_down is not None:
            top_down_up = F.interpolate(top_down, size=lat.shape[-2:], mode='nearest')
            lat = lat + top_down_up
        return self.smooth(lat)


class SimpleFPN(nn.Module):
    """
    FPN on top of ResNet-like backbone.
    Takes feature maps {C2, C3, C4, C5} → outputs {P2, P3, P4, P5, P6}.
    """
    def __init__(self, in_channels_list, out_channels=256):
        super().__init__()
        # in_channels_list: [C2_ch, C3_ch, C4_ch, C5_ch]
        self.lat_blocks = nn.ModuleList([
            LateralBlock(ch, out_channels) for ch in in_channels_list
        ])
        # P6: max-pool on P5 for even larger receptive field
        self.p6_pool = nn.MaxPool2d(1, stride=2)

    def forward(self, features):
        """
        features: OrderedDict {name: tensor} from backbone, coarsest to finest
        Returns: [P2, P3, P4, P5, P6]
        """
        # Process top-down: start from C5 (coarsest, richest)
        laterals = [lat(feat) for lat, feat in zip(self.lat_blocks, features)]

        # Top-down merging
        for i in range(len(laterals)-2, -1, -1):
            td_up = F.interpolate(laterals[i+1], size=laterals[i].shape[-2:], mode='nearest')
            laterals[i] = laterals[i] + td_up
            laterals[i] = self.lat_blocks[i].smooth(laterals[i])

        p6 = self.p6_pool(laterals[-1])  # coarser level for very large objects
        return laterals + [p6]


# Demo with dummy feature maps
c2 = torch.randn(1, 256,  100, 100)  # stride 4
c3 = torch.randn(1, 512,   50,  50)  # stride 8
c4 = torch.randn(1, 1024,  25,  25)  # stride 16
c5 = torch.randn(1, 2048,  13,  13)  # stride 32

fpn_model = SimpleFPN([256, 512, 1024, 2048], out_channels=256)
pyramids = fpn_model([c2, c3, c4, c5])

print('FPN output pyramid:')
for i, p in enumerate(pyramids):
    lvl = i + 2
    stride = 4 * (2**i)
    print(f'  P{lvl}: {tuple(p.shape)}  (stride={stride})')

## Visualize the Pyramid

Each level of the pyramid sees the same image at a different effective resolution. Small objects are detected at P2/P3 (high resolution), large objects at P5/P6 (low resolution, rich semantics).

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, (p, ax) in enumerate(zip(pyramids, axes)):
    lvl = i + 2
    # Show mean activation across 256 channels
    feat_vis = p[0].mean(0).detach().numpy()
    ax.imshow(feat_vis, cmap='viridis')
    ax.set_title(f'P{lvl}\n{p.shape[-2]}x{p.shape[-1]} (stride {4*2**i})', fontsize=9)
    ax.axis('off')
plt.suptitle('FPN Feature Pyramid (mean activation across 256 channels)', fontsize=11)
plt.tight_layout(); plt.show()

## torchvision's Built-in FPN

torchvision provides `torchvision.ops.FeaturePyramidNetwork` and integrates it directly with `fasterrcnn_resnet50_fpn`. The RPN and RoI Align automatically operate at all pyramid levels.

In [ ]:
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone

# Build ResNet50 + FPN backbone
backbone = resnet_fpn_backbone('resnet50', weights='DEFAULT', trainable_layers=3)
backbone = backbone.to(device).eval()

try:
    img = Image.open(BytesIO(requests.get('https://ultralytics.com/images/zidane.jpg',timeout=8).content)).convert('RGB')
except Exception:
    img = Image.fromarray(np.random.randint(0,255,(480,640,3),dtype=np.uint8))

img_t = T.ToTensor()(img).unsqueeze(0).to(device)

with torch.no_grad():
    feat_dict = backbone(img_t)

print('ResNet50+FPN feature pyramid:')
for k, v in feat_dict.items():
    stride = img_t.shape[-1] // v.shape[-1]
    print(f'  {k}: {tuple(v.shape)}  stride={stride}')

## FPN in Faster R-CNN: Anchors per Level

When FPN is added to Faster R-CNN:
- Each pyramid level gets its own set of anchors at a single scale
- RPN slides over all levels simultaneously
- RoI Align selects the appropriate feature level based on proposal size:

$$k = \lfloor k_0 + \log_2(\sqrt{wh}/224) \rfloor$$

A small 32×32 proposal maps to P2 (fine), a large 512×512 proposal maps to P5 (coarse).

In [ ]:
def assign_level(boxes, k0=4, min_level=2, max_level=5):
    """
    Assign each box to a FPN level based on its size.
    boxes: (N, 4) tensor [x1,y1,x2,y2]
    """
    w = boxes[:, 2] - boxes[:, 0]
    h = boxes[:, 3] - boxes[:, 1]
    area = (w * h).clamp(min=1).sqrt()
    level = (k0 + torch.log2(area / 224.0)).floor().long()
    return level.clamp(min_level, max_level)

example_boxes = torch.tensor([
    [100, 100, 132, 132],   # 32x32 small
    [50,  50,  178, 178],   # 128x128 medium
    [10,  10,  266, 266],   # 256x256 large
    [5,   5,   517, 517],   # 512x512 very large
], dtype=torch.float32)

levels = assign_level(example_boxes)
sizes = [(b[2]-b[0]).int().item() for b in example_boxes]
print('Box Size  FPN Level')
for sz, lvl in zip(sizes, levels):
    print(f'  {sz:4d}px → P{lvl}')

## FPN Impact on Detection Performance

| Model | Backbone | mAP COCO | Small AP | Medium AP | Large AP |
|-------|----------|----------|----------|-----------|----------|
| Faster R-CNN | ResNet50 | 36.4% | 19.1% | 39.5% | 48.9% |
| Faster R-CNN + FPN | ResNet50 | **39.8%** | **24.2%** | 43.5% | 50.4% |

FPN's biggest win is on **small objects** (+5.1 AP) — exactly because it gives the detector access to high-resolution features with rich semantics.

## Where FPN Appears Today

FPN is now a standard component in virtually all modern detectors and segmentation models:
- **Faster R-CNN + FPN**: torchvision default two-stage detector
- **Mask R-CNN**: adds a mask head on top of Faster R-CNN + FPN  
- **RetinaNet**: single-stage detector with FPN backbone  
- **FCOS, PANet, BiFPN (EfficientDet)**: FPN variants with richer cross-level connections

See `01-UNET.ipynb` for semantic segmentation with FPN-like skip connections, and `02-Mask-R-CNN.ipynb` for instance segmentation built on top of Faster R-CNN + FPN.